---
authors:
  - name: Tom Siegl
---

# TPLHD: Optimized Latin Hypercube Sampling


The next cell recreates the design-comparison setup from the main notebook (`num_levels_per_var`, `num_vars`, the naive and random LHS designs, and the `plot_design_matrix`/`is_latin_hypercube` helpers), since we'll need them again below to compare TPLHD against those.


In [ ]:
# This cell recreates the setup from the full-factorial-and-lhs-design notebook.

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)


def plot_design_matrix(design_matrix, fig, ax, title='', c=None):
    ax.scatter(design_matrix[:, 0], design_matrix[:, 1], zorder=10, c=c)
    ax.set_xlabel('variable 1 level idx')
    ax.set_ylabel('variable 2 level idx')
    ax.set_xticks(np.unique(design_matrix[:, 0]))
    ax.set_yticks(np.unique(design_matrix[:, 1]))
    ax.set_title(title)
    ax.grid(True)


def is_latin_hypercube(design_matrix):
    for col in range(design_matrix.shape[1]):
        if len(np.unique(design_matrix[:, col])) != design_matrix.shape[0]:
            print("Nope. This is not a latin hypercube.")
            return
            
    print("Yes. This is a latin hypercube.")


num_levels_per_var = 10
num_vars = 2


def naive_lhs(num_levels_per_var, num_vars):
    level_idx = np.array([list(range(num_levels)) for num_levels in [num_levels_per_var]*num_vars])
    lh = []
    
    for level_i in range(level_idx.shape[1]):
        lh.append([level_idx[var_i, level_i] for var_i in range(level_idx.shape[0])])

    return np.array(lh)


naive_lh = naive_lhs(num_levels_per_var, num_vars)


def random_lhs(num_levels_per_var, num_vars):
    level_idx = np.array([list(range(num_levels)) for num_levels in [num_levels_per_var]*num_vars])
    lh = []
    
    for var_i in range(level_idx.shape[0]):
        lh.append(np.random.permutation(level_idx[var_i]))

    return np.array(lh).T


random_lh = random_lhs(num_levels_per_var, num_vars)


As a reference with all the details, you can find the original paper that proposed the TPLHD algorithm for optimized latin hypercube sampling >>[here](https://doi.org/10.1002/nme.2750)<<.
You can also use the implementation below as another reference for this algorithm.

Feel free to play around with the implementation below, e.g.:
- Swap out the seed.
- Define a seed of your own.
- Change the number of levels per variable.

Go back to seed `tplhd_seed_0` and use `num_levels_per_var` from before for the number of levels per variable before you start with the tasks.

:::{important} Tasks
1. Describe how `tplhd_seed_3` is resized before the translation starts! (Write down a solution before generating the actual result using the code below.)
2. Draw the TPLHD result for `tplhd_seed_0` and `4` levels per variable! (Again, first come up with a solution, then use the code to check.)
3. Would it make sense to draw the points in the squares instead of on the lines?
4. In your drawing from task 2, mark the point that would be deleted to shrink the design to one with `3` levels per variable.
:::

:::{tip} Your answer
:class:dropdown
1. 
2. 
3. 
4. 
:::

In [ ]:
######################
### TPLHD SAMPLING ###
######################


def reshapeSeed(s, n_s, n_p_star, n_d_star, n_v):
    return s * n_d_star


def createTPLHD(s, n_s, n_p_star, n_d_star, n_v):
    translate = len(s) * n_d_star
    X = s
    
    for var_i in range(n_v):
        intermediate_seed = X.copy()
        X = [X]
        
        for div_i in range(n_d_star-1):
            intermediate_seed += 1
            intermediate_seed[:, var_i] += translate - 1
            X.append(intermediate_seed.copy())

        X = np.concatenate(X)

    return X


def resizeTPLHD(X, n_d_star, n_p, n_v):
    center = np.sum(X, axis=0) / X.shape[0]
    dists = np.sqrt((np.sum(np.square(X - center), axis=1)))

    dists_sorted_idx = np.argsort(dists)  # increasing order

    X = X.copy()[dists_sorted_idx]
    
    deleted_X = X[n_p:]
    X = X[:n_p]
    X_intermediate = X.copy()

    for var_i in range(n_v):
        deleted_sorted = np.sort(deleted_X[:, var_i])[::-1]  # decreasing order
        
        for deleted_val in deleted_sorted:
            X[X[:, var_i] > deleted_val, var_i] -= 1
    
    return X_intermediate, X


def TPLHD(n_p, n_v, s, n_s):
    """
    Implementation of https://doi.org/10.1002/nme.2750.
    
    :param n_p: Number of points to create
    :param n_v: Number of variables to create
    :param s: Seed definition array in shape (points, variables)
    :param n_s: Number of points in the seed
    :returns: Optimized latin hypercube design
    """
    n_d = (n_p/n_s)**(1/n_v)  # number divisions per variable (each division is one seed)
    n_d_star = int(np.ceil(n_d))  # number of integer divisions per variable to at least fill the levels

    n_b = None  # number of blocks
    if n_d_star > n_d:
        n_b = n_d_star**n_v  # each seed is a little more than one block (need to resize later)
    else:
        n_b = n_p / n_s  # each seed is one block (perfect fit)

    n_p_star = n_b * n_s  # number of points after seed-based construction and before resizing

    s = reshapeSeed(s, n_s, n_p_star, n_d_star, n_v)

    X = createTPLHD(s, n_s, n_p_star, n_d_star, n_v)

    X_intermediate = X
    X_resize = X
    if n_p_star > n_p:
        X_intermediate, X_resize = resizeTPLHD(X, n_d_star, n_p, n_v)

    return X, X_intermediate, X_resize

In [ ]:
tplhd_seed_0 = np.array([[0, 0]])
tplhd_seed_1 = np.array([[1, 1], [0, 0]])
tplhd_seed_2 = np.array([[0, 0], [1, 2], [2, 1]])
tplhd_seed_3 = np.array([[0, 0], [1, 2], [2, 1], [3, 3]])

tplhd_seed = tplhd_seed_0
pre_resize_lh, intermediate_lh, optimized_lh = TPLHD(num_levels_per_var, num_vars, tplhd_seed, tplhd_seed.shape[0])


is_latin_hypercube(optimized_lh)


# plot
fig = plt.figure(figsize=(16, 5))

ax1 = fig.add_subplot(131)
plot_design_matrix(tplhd_seed, fig, ax1, title='seed')

ax2 = fig.add_subplot(132)
plot_design_matrix(pre_resize_lh, fig, ax2, title='TPLHD before resizing', c='red')
ax2.scatter(intermediate_lh[:, 0], intermediate_lh[:, 1], zorder=20)

ax2 = fig.add_subplot(133)
plot_design_matrix(optimized_lh, fig, ax2, title='TPLHD')

plt.show()

For the phi criterion we just use the existing implementation from pyLHD:
- [GitHub](https://github.com/toledo60/pyLHD)
- [Documentation](https://pylhd.netlify.app/notebooks/intro_pylhd)

This package also comes with a collection of many more optimized latin hypercube sampling algorithms.

:::{important} Tasks
1. What is the meaning of the output of the following code cell w.r.t. comparing the sampling methods?
:::

:::{tip} Your answer
:class:dropdown
1. 
:::

In [ ]:
import pyLHD

print(f'Naive:          {pyLHD.phi_p(naive_lh, p=15, q=2):.3f}')
print(f'Random:         {pyLHD.phi_p(random_lh, p=15, q=2):.3f}')
print(f'TPLHD:          {pyLHD.phi_p(optimized_lh, p=15, q=2):.3f}')